# ZuCo text + EEG sentiment experiments

This notebook is only the Colab runner. The experiment logic lives in the Python files in the repository. It mounts Drive, prepares paths, builds the reusable classical-feature cache, validates alignment, runs smoke tests, prepares one persistent LaBSE copy, launches versioned experiment suites, and displays the results saved to Drive.

The first full suite compares frozen and fine-tuned LaBSE text baselines, EEG only, concatenation, gated fusion, shuffled EEG, and random-noise EEG over three seeds and five sentence-level folds. Later sections run matched controlled diagnostics with aligned, shuffled, noise, and forced-zero EEG contributions for both fine-tuned and frozen LaBSE. Finished setup/seed files are skipped when a run is resumed.

## 1. Runtime and Drive

This section checks whether Colab has a GPU and mounts Drive. The original ZuCo files, the reusable feature cache, and every experiment result remain on Drive.

In [ ]:
import os
import torch
from google.colab import drive

drive.mount('/content/drive')
print('torch:', torch.__version__)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'not available')

## 2. Repository and dependencies

This section clones the repository on the first run and pulls the latest commit on later runs. It then installs the Python requirements. No experiment outputs are written into the repository.

In [ ]:
REPO_URL = 'https://github.com/parmisbathaeiyan/zuco-multimodal-sentiment.git'
REPO_DIR = '/content/zuco-multimodal-sentiment'

if not os.path.exists(REPO_DIR):
    !git clone $REPO_URL $REPO_DIR
else:
    !git -C $REPO_DIR pull --ff-only
%cd $REPO_DIR
!pip install -q -r requirements.txt

## 3. Paths and run version

This section defines the Drive data, cached-artifact, and output locations. The fixed labels CSV is in the main Thesis data folder, while the subject `.mat` files are in its `zuco_og_raw` subfolder. `FEATURE_TAG` versions the extracted EEG definition. `RUN_TAG` versions one experiment configuration. Re-running the same `RUN_TAG` resumes missing work; change it when the architecture or training settings change. If the feature cache was created with the earlier notebook, it is moved into `CachedArtifacts` without being extracted again.

In [ ]:
import shutil

THESIS_ROOT = '/content/drive/MyDrive/Thesis'
DATA_ROOT = f'{THESIS_ROOT}/Data'
MAT_DIR = f'{DATA_ROOT}/zuco_og_raw'
LABELS_CSV = f'{DATA_ROOT}/zuco_sentiment_labels_task1_fixed.csv'

ARTIFACTS_ROOT = f'{THESIS_ROOT}/CachedArtifacts/zuco_multimodal_sentiment'
FEATURE_TAG = 'classical_v2_normalized_line_length'
FEATURES_DIR = f'{ARTIFACTS_ROOT}/eeg_features/{FEATURE_TAG}'
MODEL_DRIVE_DIR = f'{ARTIFACTS_ROOT}/models/LaBSE'
MODEL_RUNTIME_DIR = '/content/cached_artifacts/LaBSE'

RESULTS_BASE = f'{THESIS_ROOT}/Results/zuco_multimodal_sentiment'
RUN_TAG = 'v1_full'  # change this for a genuinely new experiment version
RUN_DIR = f'{RESULTS_BASE}/{RUN_TAG}'

SETUPS = [
    'text_frozen',
    'text_finetune',
    'eeg_only',
    'concat_finetune',
    'gated_finetune',
    'gated_shuffled_finetune',
    'gated_noise_finetune',
]
SEEDS = [42, 52, 62]

legacy_features = f'{DATA_ROOT}/zuco_classical_features/{FEATURE_TAG}'
if not os.path.exists(FEATURES_DIR) and os.path.exists(legacy_features):
    os.makedirs(os.path.dirname(FEATURES_DIR), exist_ok=True)
    shutil.move(legacy_features, FEATURES_DIR)
    print('moved existing feature cache into CachedArtifacts')
else:
    os.makedirs(FEATURES_DIR, exist_ok=True)
os.makedirs(RESULTS_BASE, exist_ok=True)
assert os.path.exists(LABELS_CSV), f'Missing labels file: {LABELS_CSV}'
print('labels:', LABELS_CSV)
print('feature cache:', FEATURES_DIR)
print('model cache:', MODEL_DRIVE_DIR)
print('run results:', RUN_DIR)

## 4. Build or resume the EEG feature cache

This section reads each large `results*_SR.mat` file and saves one much smaller subject `.npz` file to the versioned Drive cache. It is the slow data-preparation step, but completed subjects are skipped on later runs. The default uses duration-normalized line length.

In [ ]:
!python extract_features.py \
    --mat-dir "$MAT_DIR" \
    --labels-csv "$LABELS_CSV" \
    --out-dir "$FEATURES_DIR" \
    --line-length normalized

## 5. Validate text/EEG alignment

This section checks feature widths, sentence IDs, labels, subject counts, missing recordings, and the exact names of any feature columns that are missing throughout the dataset. The flat Cz reference channel (`ch104`) is excluded during loading, so the model should receive 104 channels and 2,496 features.

In [ ]:
!python inspect_data.py --labels-csv "$LABELS_CSV" --features-dir "$FEATURES_DIR"

## 6. Short smoke test

This optional run trains the EEG branch for one epoch over two folds. It catches data-shape and saving errors quickly before the expensive LaBSE suite. Its output uses a separate run tag and does not mix with the full results.

In [ ]:
SMOKE_TAG = f'{RUN_TAG}_smoke_no_cz'
!python run.py \
    --labels-csv "$LABELS_CSV" \
    --features-dir "$FEATURES_DIR" \
    --results-base "$RESULTS_BASE" \
    --run-tag "$SMOKE_TAG" \
    --setups eeg_only \
    --seeds 42 \
    --n-folds 2 \
    --epochs 1 \
    --patience 1 \
    --bootstrap-samples 200

from IPython.display import Markdown, display

DIAGNOSTIC_SMOKE_DIR = f'{RESULTS_BASE}/{DIAGNOSTIC_SMOKE_TAG}'
for table in ['control_comparisons.md', 'diagnostics.md']:
    path = os.path.join(DIAGNOSTIC_SMOKE_DIR, 'tables', table)
    if os.path.exists(path):
        display(Markdown(open(path).read()))

smoke_metadata = os.path.join(
    DIAGNOSTIC_SMOKE_DIR, 'tables', 'diagnostic_metadata.json'
)
if os.path.exists(smoke_metadata):
    print(open(smoke_metadata).read())

## 7. Prepare the persistent LaBSE model

The full suite needs LaBSE. The first run downloads about 1.90 GB into `CachedArtifacts`; later sessions reuse that Drive copy without downloading it again. The saved model is copied to temporary Colab storage once per session because repeatedly loading large weights directly from mounted Drive would slow every fold.

In [ ]:
from huggingface_hub import snapshot_download

MODEL_REPO = 'sentence-transformers/LaBSE'
MODEL_FILES = [
    'config.json',
    'model.safetensors',
    'special_tokens_map.json',
    'tokenizer.json',
    'tokenizer_config.json',
    'vocab.txt',
]
model_ready = os.path.exists(os.path.join(MODEL_DRIVE_DIR, 'model.safetensors'))
if not model_ready:
    print('LaBSE requires about 1.90 GB of network download and Drive storage.')
    snapshot_download(
        repo_id=MODEL_REPO,
        local_dir=MODEL_DRIVE_DIR,
        allow_patterns=MODEL_FILES,
    )
else:
    print('reusing LaBSE from Drive:', MODEL_DRIVE_DIR)

print('copying LaBSE to the temporary Colab disk for faster fold loading')
shutil.copytree(
    MODEL_DRIVE_DIR,
    MODEL_RUNTIME_DIR,
    dirs_exist_ok=True,
    ignore=shutil.ignore_patterns('.cache'),
)
assert os.path.exists(os.path.join(MODEL_RUNTIME_DIR, 'model.safetensors'))
print('runtime model:', MODEL_RUNTIME_DIR)

## 8. Short text + multimodal smoke test

This run checks the paths that the EEG-only smoke test cannot cover: loading the cached LaBSE model, fine-tuning the text encoder, gated text–EEG fusion, and GPU memory use. It runs the matching text and gated setups for one epoch over two folds, then saves them under a separate smoke tag.

In [ ]:
MULTIMODAL_SMOKE_TAG = f'{RUN_TAG}_smoke_multimodal'
!python run.py \
    --labels-csv "$LABELS_CSV" \
    --features-dir "$FEATURES_DIR" \
    --results-base "$RESULTS_BASE" \
    --run-tag "$MULTIMODAL_SMOKE_TAG" \
    --model-name "$MODEL_RUNTIME_DIR" \
    --setups text_finetune gated_finetune \
    --seeds 42 \
    --n-folds 2 \
    --epochs 1 \
    --patience 1 \
    --bootstrap-samples 200

## 9. Full versioned experiment suite

This section runs every setup in `SETUPS` over every seed in `SEEDS` using five sentence-level folds. Each setup/seed JSON is saved as soon as it finishes. The full suite is substantial, but it can be interrupted and resumed by rerunning this cell with the same `RUN_TAG`.

In [ ]:
SETUP_ARGS = ' '.join(SETUPS)
SEED_ARGS = ' '.join(str(seed) for seed in SEEDS)
!python run.py \
    --labels-csv "$LABELS_CSV" \
    --features-dir "$FEATURES_DIR" \
    --results-base "$RESULTS_BASE" \
    --run-tag "$RUN_TAG" \
    --model-name "$MODEL_RUNTIME_DIR" \
    --setups $SETUP_ARGS \
    --seeds $SEED_ARGS \
    --n-folds 5 \
    --epochs 12 \
    --patience 4

## 10. Saved summary and plots

This section reads the final artifacts from Drive. The table reports mean out-of-fold accuracy and macro-F1 across seeds, plus paired bootstrap deltas for fusion models. The figures compare scores and confusion matrices.

In [ ]:
from IPython.display import Image, Markdown, display

summary_path = os.path.join(RUN_DIR, 'tables', 'summary.md')
if os.path.exists(summary_path):
    display(Markdown(open(summary_path).read()))
else:
    print('No summary yet:', summary_path)

for name in ['scores.png', 'confusions.png']:
    path = os.path.join(RUN_DIR, 'plots', name)
    if os.path.exists(path):
        display(Image(path))

## 11. Rebuild reports without retraining

Use this cell after changing only plotting or reporting code. It regenerates the summary tables and figures from the saved per-seed JSON files and does not load or train a model.

In [ ]:
!python plot_results.py --run-dir "$RUN_DIR"

## 12. Controlled-diagnostic smoke test

This optional one-seed, two-fold run checks the new matched-initialization and forced-zero paths before the longer diagnostic experiment. Both gated models have the same modules, initialization seed, dropout schedule, and text fine-tuning path. The zero setup computes the EEG branch but forces its contribution to exactly zero.

In [ ]:
DIAGNOSTIC_RUN_TAG = 'v2_controlled_diagnostics'
DIAGNOSTIC_RUN_DIR = f'{RESULTS_BASE}/{DIAGNOSTIC_RUN_TAG}'
DIAGNOSTIC_SMOKE_TAG = f'{DIAGNOSTIC_RUN_TAG}_smoke'

!python run.py \
    --labels-csv "$LABELS_CSV" \
    --features-dir "$FEATURES_DIR" \
    --results-base "$RESULTS_BASE" \
    --run-tag "$DIAGNOSTIC_SMOKE_TAG" \
    --model-name "$MODEL_RUNTIME_DIR" \
    --setups gated_finetune gated_zero_finetune \
    --seeds 42 \
    --n-folds 2 \
    --epochs 1 \
    --patience 1 \
    --bootstrap-samples 200

## 13. Fine-tuned matched-control experiment

This section reruns the fine-tuned comparison under a new version. Aligned, shuffled, noise, and zero gated models start from matching task-head weights for each seed and fold. The saved JSON files include complete gate values, embedding and contribution norms, and held-out logits with and without the EEG contribution. Re-run this cell with the same tag to resume after a disconnect.

In [ ]:
FINETUNE_DIAGNOSTIC_SETUPS = [
    'text_finetune',
    'gated_finetune',
    'gated_shuffled_finetune',
    'gated_noise_finetune',
    'gated_zero_finetune',
]
FINETUNE_DIAGNOSTIC_ARGS = ' '.join(FINETUNE_DIAGNOSTIC_SETUPS)
SEED_ARGS = ' '.join(str(seed) for seed in SEEDS)

!python run.py \
    --labels-csv "$LABELS_CSV" \
    --features-dir "$FEATURES_DIR" \
    --results-base "$RESULTS_BASE" \
    --run-tag "$DIAGNOSTIC_RUN_TAG" \
    --model-name "$MODEL_RUNTIME_DIR" \
    --setups $FINETUNE_DIAGNOSTIC_ARGS \
    --seeds $SEED_ARGS \
    --n-folds 5 \
    --epochs 12 \
    --patience 4

## 14. Frozen-text matched-control experiment

This section tests whether fine-tuned LaBSE was overpowering EEG. It compares frozen text only with aligned, shuffled, noise, and zero gated models while preserving the same matched-control protocol. It uses the same diagnostic run folder, so completed fine-tuned results are skipped and retained.

In [ ]:
FROZEN_DIAGNOSTIC_SETUPS = [
    'text_frozen',
    'gated_frozen',
    'gated_shuffled_frozen',
    'gated_noise_frozen',
    'gated_zero_frozen',
]
FROZEN_DIAGNOSTIC_ARGS = ' '.join(FROZEN_DIAGNOSTIC_SETUPS)

!python run.py \
    --labels-csv "$LABELS_CSV" \
    --features-dir "$FEATURES_DIR" \
    --results-base "$RESULTS_BASE" \
    --run-tag "$DIAGNOSTIC_RUN_TAG" \
    --model-name "$MODEL_RUNTIME_DIR" \
    --setups $FROZEN_DIAGNOSTIC_ARGS \
    --seeds $SEED_ARGS \
    --n-folds 5 \
    --epochs 12 \
    --patience 4

## 15. Controlled-diagnostic results

This section displays the main scores, direct aligned-versus-control comparisons, and modality diagnostics. The metadata file records how many seed/fold groups passed the matched-initialization fingerprint check.

In [ ]:
import os
from IPython.display import Image, Markdown, display

DIAGNOSTIC_RUN_DIR = (
    '/content/drive/MyDrive/Thesis/Results/'
    'zuco_multimodal_sentiment/v2_controlled_diagnostics'
)

for table in ['summary.md', 'control_comparisons.md', 'diagnostics.md']:
    path = os.path.join(DIAGNOSTIC_RUN_DIR, 'tables', table)
    if os.path.exists(path):
        display(Markdown(open(path).read()))
    else:
        print('Missing:', path)

metadata_path = os.path.join(
    DIAGNOSTIC_RUN_DIR, 'tables', 'diagnostic_metadata.json'
)
if os.path.exists(metadata_path):
    print(open(metadata_path).read())

for name in ['scores.png', 'confusions.png']:
    path = os.path.join(DIAGNOSTIC_RUN_DIR, 'plots', name)
    if os.path.exists(path):
        display(Image(path))